# Iteration 1: Data Audit + Baseline

Goal: create the first reproducible baseline for the March Madness classification challenge.

This notebook does four things:
- loads and inspects the matchup data
- derives the target at the team-opponent row level
- performs a first leakage audit
- benchmarks simple baseline models

Expected data location:
- place the challenge CSVs in `Challenge-1/data/`
- at minimum this notebook needs `Tournament Matchups.csv`

If the column suggestions are wrong after loading the file, update only the `CONFIG` cell and rerun from there.


In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 27
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")


In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Challenge-1").exists():
            return candidate
    raise FileNotFoundError("Could not find the IS-455 project root from the current notebook location.")


PROJECT_ROOT = find_project_root(Path.cwd())
CHALLENGE_DIR = PROJECT_ROOT / "Challenge-1"
DATA_DIR = CHALLENGE_DIR / "data"
ML_KIT_DIR = PROJECT_ROOT / "ML-Pipeline-Kit"

if str(ML_KIT_DIR) not in sys.path:
    sys.path.append(str(ML_KIT_DIR))

import ml_library as ml

SEARCH_ROOTS = [DATA_DIR, CHALLENGE_DIR, PROJECT_ROOT]


def normalize_token(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def discover_csvs(search_roots: list[Path]) -> pd.DataFrame:
    records = []
    for root in search_roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*.csv")):
            try:
                relative_parent = path.parent.relative_to(PROJECT_ROOT)
            except ValueError:
                relative_parent = path.parent
            records.append(
                {
                    "name": path.name,
                    "parent": str(relative_parent),
                    "path": str(path),
                }
            )
    if not records:
        return pd.DataFrame(columns=["name", "parent", "path"])
    return (
        pd.DataFrame(records)
        .drop_duplicates(subset=["path"])
        .sort_values(["name", "parent", "path"])
        .reset_index(drop=True)
    )


def find_file(file_table: pd.DataFrame, target_names: list[str]) -> Path:
    normalized_targets = {normalize_token(name) for name in target_names}
    if file_table.empty:
        raise FileNotFoundError(
            "No CSV files were found under the search roots. Put the challenge CSVs in Challenge-1/data/."
        )

    match_mask = file_table["name"].map(normalize_token).isin(normalized_targets)
    if not match_mask.any():
        searched = "\n".join(f"- {root}" for root in SEARCH_ROOTS)
        available = file_table["name"].drop_duplicates().sort_values().tolist()
        raise FileNotFoundError(
            "Could not find Tournament Matchups.csv.\n"
            "Searched in:\n"
            f"{searched}\n\n"
            "Available CSV files:\n"
            + "\n".join(f"- {name}" for name in available)
        )

    return Path(file_table.loc[match_mask, "path"].iloc[0])


def suggest_column(df: pd.DataFrame, keywords: list[str], exclude: list[str] | None = None) -> str | None:
    exclude = set(exclude or [])
    normalized_cols = {col: normalize_token(col) for col in df.columns}
    exact_targets = {normalize_token(keyword) for keyword in keywords}

    exact_matches = [
        col for col, token in normalized_cols.items() if token in exact_targets and col not in exclude
    ]
    if exact_matches:
        return exact_matches[0]

    partial_matches = []
    for col, token in normalized_cols.items():
        if col in exclude:
            continue
        if any(normalize_token(keyword) in token for keyword in keywords):
            partial_matches.append(col)
    return partial_matches[0] if partial_matches else None


def suggest_game_id_cols(df: pd.DataFrame) -> list[str]:
    candidates = []
    keyword_groups = [
        ["gameid", "matchupid", "contestid", "eventid"],
        ["year", "season"],
        ["date", "gamedate"],
        ["round"],
        ["region", "bracket", "site", "location"],
    ]
    for keywords in keyword_groups:
        col = suggest_column(df, keywords, exclude=candidates)
        if col:
            candidates.append(col)
    return candidates


def parse_seed(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(series.astype(str).str.extract(r"(\\d+)")[0], errors="coerce")


print(f"Project root: {PROJECT_ROOT}")
print(f"Challenge directory: {CHALLENGE_DIR}")
print(f"Preferred data directory: {DATA_DIR}")


Project root: C:\Users\GeorgeColinRamsay\Documents\Github\IS-455
Challenge directory: C:\Users\GeorgeColinRamsay\Documents\Github\IS-455\Challenge-1
Preferred data directory: C:\Users\GeorgeColinRamsay\Documents\Github\IS-455\Challenge-1\data


In [3]:
def build_team_opponent_view(df: pd.DataFrame, config: dict) -> tuple[pd.DataFrame, int, list[str]]:
    required_keys = ["team_col", "score_col"]
    missing_keys = [key for key in required_keys if not config.get(key)]
    if missing_keys:
        raise ValueError(f"Update CONFIG before continuing. Missing: {missing_keys}")

    game_id_cols = [col for col in config.get("game_id_cols", []) if col]
    if not game_id_cols:
        raise ValueError("CONFIG['game_id_cols'] must identify one game before pairing team rows.")

    required_cols = list(dict.fromkeys(game_id_cols + [config["team_col"], config["score_col"]]))
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"These configured columns were not found: {missing_cols}")

    working = df.copy()
    group_sizes = working.groupby(game_id_cols, dropna=False)[config["team_col"]].transform("size")
    paired_source = working.loc[group_sizes == 2].copy()
    excluded_rows = int(len(working) - len(paired_source))

    if paired_source.empty:
        raise ValueError(
            "No two-row games were found with the current CONFIG['game_id_cols']. "
            "Inspect the schema summary above and adjust the game id columns."
        )

    paired_source = paired_source.sort_values(game_id_cols + [config["team_col"]]).copy()
    paired_source["row_in_game"] = paired_source.groupby(game_id_cols, dropna=False).cumcount()

    opponent_view = paired_source.copy()
    opponent_view["row_in_game"] = 1 - opponent_view["row_in_game"]

    paired = paired_source.merge(
        opponent_view,
        on=game_id_cols + ["row_in_game"],
        suffixes=("", "_opp"),
        validate="1:1",
    )

    score_col = config["score_col"]
    paired["WIN"] = (
        pd.to_numeric(paired[score_col], errors="coerce")
        > pd.to_numeric(paired[f"{score_col}_opp"], errors="coerce")
    ).astype(int)

    seed_col = config.get("seed_col")
    if seed_col and seed_col in paired.columns and f"{seed_col}_opp" in paired.columns:
        paired["seed_num"] = parse_seed(paired[seed_col])
        paired["seed_opp_num"] = parse_seed(paired[f"{seed_col}_opp"])
        paired["seed_diff"] = paired["seed_num"] - paired["seed_opp_num"]

    year_col = config.get("year_col")
    if year_col and year_col in paired.columns:
        paired["year_num"] = pd.to_numeric(paired[year_col], errors="coerce")

    leakage_terms = ("score", "points", "result", "win", "loss", "margin", "mov", "round")
    leakage_cols = []
    for col in paired.columns:
        token = normalize_token(col)
        if any(term in token for term in leakage_terms):
            leakage_cols.append(col)

    if config.get("round_col"):
        leakage_cols.extend([config["round_col"], f"{config['round_col']}_opp"])

    leakage_cols = sorted({col for col in leakage_cols if col in paired.columns})
    return paired, excluded_rows, leakage_cols


def choose_split(X: pd.DataFrame, y: pd.Series, groups: pd.Series | None = None) -> tuple[np.ndarray, np.ndarray, str]:
    index = np.arange(len(X))
    if groups is not None and pd.Series(groups).nunique(dropna=True) >= 5:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))
        return np.asarray(train_idx), np.asarray(test_idx), "GroupShuffleSplit(test_size=0.20) by year"

    train_idx, test_idx = train_test_split(
        index,
        test_size=0.20,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    return np.asarray(train_idx), np.asarray(test_idx), "Stratified train_test_split(test_size=0.20)"


def choose_cv(y_train: pd.Series, groups_train: pd.Series | None = None):
    if groups_train is not None and pd.Series(groups_train).nunique(dropna=True) >= 3:
        n_splits = min(5, pd.Series(groups_train).nunique(dropna=True))
        return GroupKFold(n_splits=n_splits), f"GroupKFold({n_splits}) by year"

    min_class_count = int(pd.Series(y_train).value_counts().min())
    n_splits = min(5, min_class_count)
    if n_splits < 2:
        raise ValueError("Not enough observations per class for cross-validation.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE), f"StratifiedKFold({n_splits})"


def make_pipeline(feature_cols: list[str], estimator, scale_numeric: bool) -> Pipeline:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(steps=numeric_steps), feature_cols),
        ],
        remainder="drop",
    )
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )


def evaluate_models(
    models: list[tuple[str, object, bool]],
    feature_cols: list[str],
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    groups_train: pd.Series | None = None,
) -> tuple[pd.DataFrame, dict[str, Pipeline], str]:
    cv, cv_name = choose_cv(y_train=y_train, groups_train=groups_train)
    fitted_models = {}
    rows = []

    for model_name, estimator, scale_numeric in models:
        pipeline = make_pipeline(feature_cols, estimator, scale_numeric=scale_numeric)

        if isinstance(cv, GroupKFold):
            cv_scores = cross_val_score(
                pipeline,
                X_train,
                y_train,
                cv=cv,
                groups=groups_train,
                scoring="accuracy",
            )
        else:
            cv_scores = cross_val_score(
                pipeline,
                X_train,
                y_train,
                cv=cv,
                scoring="accuracy",
            )

        pipeline.fit(X_train, y_train)
        fitted_models[model_name] = pipeline

        train_pred = pipeline.predict(X_train)
        test_pred = pipeline.predict(X_test)

        rows.append(
            {
                "model": model_name,
                "train_accuracy": accuracy_score(y_train, train_pred),
                "cv_accuracy_mean": float(np.mean(cv_scores)),
                "cv_accuracy_std": float(np.std(cv_scores)),
                "test_accuracy": accuracy_score(y_test, test_pred),
            }
        )

    results = (
        pd.DataFrame(rows)
        .sort_values(["test_accuracy", "cv_accuracy_mean"], ascending=False)
        .reset_index(drop=True)
    )
    return results, fitted_models, cv_name


In [4]:
challenge_files = discover_csvs(SEARCH_ROOTS)
display(challenge_files)

MATCHUP_FILE = find_file(
    challenge_files,
    ["Tournament Matchups.csv", "Tournament_Matchups.csv", "tournament matchups.csv"],
)

matchups_raw = pd.read_csv(MATCHUP_FILE)

print(f"Loaded: {MATCHUP_FILE}")
print(f"Shape: {matchups_raw.shape[0]:,} rows x {matchups_raw.shape[1]:,} columns")

display(matchups_raw.head())
display(
    pd.DataFrame(
        {
            "column": matchups_raw.columns,
            "dtype": matchups_raw.dtypes.astype(str).values,
            "missing": matchups_raw.isna().sum().values,
            "unique": matchups_raw.nunique(dropna=False).values,
        }
    )
)


,name,parent,path
0,2016-06-29.csv,.venv\Lib\site-packages\statsmodels\tsa\states...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
1,2016-07-29.csv,.venv\Lib\site-packages\statsmodels\tsa\states...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
2,ARMLEConstantPredict.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
3,AROLSConstantPredict.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
4,AROLSNoConstantPredict.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
...,...,...,...
257,yhat_css_nc.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
258,yhat_exact_c.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
259,yhat_exact_nc.csv,.venv\Lib\site-packages\statsmodels\tsa\tests\...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...
260,yhat_mnlogit.csv,.venv\Lib\site-packages\statsmodels\discrete\t...,C:\Users\GeorgeColinRamsay\Documents\Github\IS...


FileNotFoundError: Could not find Tournament Matchups.csv.
Searched in:
- C:\Users\GeorgeColinRamsay\Documents\Github\IS-455\Challenge-1\data
- C:\Users\GeorgeColinRamsay\Documents\Github\IS-455\Challenge-1
- C:\Users\GeorgeColinRamsay\Documents\Github\IS-455

Available CSV files:
- 2016-06-29.csv
- 2016-07-29.csv
- ARMLEConstantPredict.csv
- AROLSConstantPredict.csv
- AROLSNoConstantPredict.csv
- E6.csv
- E6_jmulti.csv
- Stocks.csv
- Table1_Summary_Statistics.csv
- Table2_Correlation_Matrix.csv
- Table3_High_vs_Low_Performance.csv
- Table4_Regression_Coefficients.csv
- anes96.csv
- arima111_forecasts.csv
- arima212_forecast.csv
- autos.csv
- autos_exog.csv
- autos_predict.csv
- bds_data.csv
- bds_results.csv
- binary_constrict.csv
- bmt.csv
- bmt_results.csv
- bootleg.csv
- breast_cancer.csv
- cancer.csv
- cataneo2.csv
- ccard.csv
- cfa_tvpvar_Omega_11.csv
- cfa_tvpvar_Omega_22.csv
- cfa_tvpvar_S10.csv
- cfa_tvpvar_Si0.csv
- cfa_tvpvar_beta.csv
- cfa_tvpvar_invP.csv
- cfa_tvpvar_poste

## Schema Review

The next cell suggests likely columns based on names. If any suggestion is wrong, edit `CONFIG` before running the pairing cell.

The most important field is `game_id_cols`: it must identify one game before pairing the two team rows together.


In [ ]:
SCHEMA_SUGGESTIONS = {
    "year_col": suggest_column(matchups_raw, ["year", "season"]),
    "round_col": suggest_column(matchups_raw, ["round"]),
    "team_col": suggest_column(matchups_raw, ["team", "teamname", "school", "program"]),
    "score_col": suggest_column(matchups_raw, ["score", "points", "pts"]),
    "seed_col": suggest_column(matchups_raw, ["seed"]),
}
SCHEMA_SUGGESTIONS["game_id_cols"] = suggest_game_id_cols(matchups_raw)

display(pd.Series(SCHEMA_SUGGESTIONS, name="suggested_value"))

CONFIG = {
    "year_col": SCHEMA_SUGGESTIONS["year_col"],
    "round_col": SCHEMA_SUGGESTIONS["round_col"],
    "team_col": SCHEMA_SUGGESTIONS["team_col"],
    "score_col": SCHEMA_SUGGESTIONS["score_col"],
    "seed_col": SCHEMA_SUGGESTIONS["seed_col"],
    "game_id_cols": SCHEMA_SUGGESTIONS["game_id_cols"],
}

if CONFIG["game_id_cols"]:
    group_size_preview = (
        matchups_raw.groupby(CONFIG["game_id_cols"], dropna=False)
        .size()
        .value_counts()
        .sort_index()
        .rename_axis("rows_per_group")
        .to_frame("num_groups")
    )
    display(group_size_preview)
else:
    print("Update CONFIG['game_id_cols'] before continuing.")

CONFIG


In [ ]:
paired_games, excluded_rows, leakage_cols = build_team_opponent_view(matchups_raw, CONFIG)

print(f"Rows excluded because the configured game id did not produce exactly two team rows: {excluded_rows}")
print(f"Paired modeling table shape: {paired_games.shape[0]:,} rows x {paired_games.shape[1]:,} columns")

label_balance = paired_games["WIN"].value_counts(normalize=True).rename("share").sort_index()
display(label_balance.to_frame())

print("Leakage columns flagged for exclusion:")
display(pd.DataFrame({"leakage_column": leakage_cols}))

preview_cols = [
    col
    for col in [
        *CONFIG["game_id_cols"],
        CONFIG["team_col"],
        f"{CONFIG['team_col']}_opp",
        CONFIG["score_col"],
        f"{CONFIG['score_col']}_opp",
        CONFIG["seed_col"],
        f"{CONFIG['seed_col']}_opp" if CONFIG["seed_col"] else None,
        "WIN",
    ]
    if col and col in paired_games.columns
]

display(paired_games[preview_cols].head(10))


In [ ]:
candidate_baseline_features = [
    "seed_num",
    "seed_opp_num",
    "seed_diff",
    "year_num",
]
BASELINE_FEATURES = [col for col in candidate_baseline_features if col in paired_games.columns]

if not BASELINE_FEATURES:
    raise ValueError(
        "No baseline features were created. Check whether seed and year columns were detected correctly, "
        "or add a small set of safe pre-game features here after inspecting the schema."
    )

print("Baseline features:", BASELINE_FEATURES)

baseline_df = paired_games[BASELINE_FEATURES + ["WIN"]].copy()
display(baseline_df.head())
display(ml.univariate(baseline_df[BASELINE_FEATURES]))


In [ ]:
X = paired_games[BASELINE_FEATURES].copy()
y = paired_games["WIN"].copy()

groups = None
if "year_num" in paired_games.columns and paired_games["year_num"].notna().any():
    groups = paired_games["year_num"]

train_idx, test_idx, split_name = choose_split(X, y, groups=groups)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy() if isinstance(groups, pd.Series) else None

print(f"Split strategy: {split_name}")
print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")

models = [
    ("DummyClassifier", DummyClassifier(strategy="prior"), False),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, solver="liblinear", random_state=RANDOM_STATE),
        True,
    ),
    (
        "DecisionTree_depth3",
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=RANDOM_STATE),
        False,
    ),
]

results, fitted_models, cv_name = evaluate_models(
    models=models,
    feature_cols=BASELINE_FEATURES,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    groups_train=groups_train,
)

print(f"CV strategy: {cv_name}")
display(
    results.style.format(
        {
            "train_accuracy": "{:.3f}",
            "cv_accuracy_mean": "{:.3f}",
            "cv_accuracy_std": "{:.3f}",
            "test_accuracy": "{:.3f}",
        }
    )
)


In [ ]:
best_model_name = results.loc[0, "model"]
best_model = fitted_models[best_model_name]

test_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"{best_model_name} confusion matrix")
plt.show()

print(classification_report(y_test, test_pred, digits=3))

test_view = paired_games.iloc[test_idx].copy().reset_index(drop=True)
test_view["actual"] = y_test.reset_index(drop=True)
test_view["predicted"] = pd.Series(test_pred)
if hasattr(best_model, "predict_proba"):
    test_view["pred_win"] = best_model.predict_proba(X_test)[:, 1]

error_cols = [
    col
    for col in [
        *CONFIG["game_id_cols"],
        CONFIG["team_col"],
        f"{CONFIG['team_col']}_opp",
        CONFIG["score_col"],
        f"{CONFIG['score_col']}_opp",
        "seed_num",
        "seed_opp_num",
        "seed_diff",
        "actual",
        "predicted",
        "pred_win",
    ]
    if col and col in test_view.columns
]

display(
    test_view.loc[test_view["actual"] != test_view["predicted"], error_cols]
    .sort_values("pred_win", ascending=False, na_position="last")
    .head(25)
)


In [ ]:
iteration_summary = pd.DataFrame(
    [
        {
            "Iteration": "iter_1",
            "What Changed": "Built a matchup-only baseline with target creation, leakage audit, and seed-based starter features.",
            "Best Validation Score": results.loc[0, "cv_accuracy_mean"],
            "Test Score": results.loc[0, "test_accuracy"],
            "Keep / Drop Next Time": "Keep the split and pairing logic; next add safe pre-game team-strength tables.",
        }
    ]
)

display(
    iteration_summary.style.format(
        {
            "Best Validation Score": "{:.3f}",
            "Test Score": "{:.3f}",
        }
    )
)
